In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split

In [5]:
df = pd.read_csv("Transformer_Data_Merged.csv")
df = df.fillna(0)

df["Scheduled_Hours"] = df["Scheduled_Hours"].replace(0, 1)
df["MTBF_Hours"] = df["MTBF_Hours"].replace(0, 1)
df["MTTR_Hours"] = df["MTTR_Hours"].replace(0, 1)

In [6]:
df["Failure_Rate"] = 1.0 / df["MTBF_Hours"]
df["Repair_Efficiency"] = 1.0 / df["MTTR_Hours"]
df["Utilization_Stress"] = df["Utilization_Rate"] * df["Scheduled_Hours"]
df["Downtime_Ratio"] = df["Downtime_Duration"] / df["Scheduled_Hours"]

df["Maintenance_Cost_Rate"] = df["Maintenance_Parts_Cost"] / (df["Scheduled_Hours"] + 1)
df["Energy_Cost_Rate"] = df["Energy_Consumption_kWh"] / (df["Output_Quantity"] + 1)
df["Reject_Rate"] = df["Reject_Quantity"] / (df["Output_Quantity"] + 1)

df["Health_Index"] = (
    0.35 * df["Uptime_Percentage"] +
    0.25 * (1 - df["Failure_Rate"]) +
    0.20 * (1 - df["Downtime_Ratio"]) +
    0.20 * (1 - df["Reject_Rate"])
)

In [7]:
state_features = [
    "Health_Index", "Failure_Rate", "Repair_Efficiency",
    "Utilization_Stress", "Downtime_Ratio",
    "Maintenance_Cost_Rate", "Energy_Cost_Rate",
    "Reject_Rate", "Number_of_Breakdowns"
]

states = df[state_features].values

In [8]:
df["Reward"] = df["Net_Profit"]  # FIXED (no double subtraction)
dataset_rewards = df["Reward"].values

In [9]:
scaler = MinMaxScaler()
states = scaler.fit_transform(states)

In [10]:
X_train, X_test = train_test_split(states, test_size=0.2, random_state=42)

In [11]:
num_actions = 3
state_dim = X_train.shape[1]

weights = np.zeros((num_actions, state_dim))

alpha = 0.01
gamma = 0.95
epsilon = 0.2
episodes = 500

In [12]:
def get_q_values(state):
    return np.dot(weights, state)

def choose_action(state):
    if np.random.rand() < epsilon:
        return np.random.randint(num_actions)
    return np.argmax(get_q_values(state))

In [13]:
def apply_probabilistic_change(value, prob_inc, prob_dec, step=0.05):
    r = np.random.rand()
    if r < prob_inc:
        value += step
    elif r < prob_inc + prob_dec:
        value -= step
    return value

def environment_step(state, action):
    next_state = state.copy()

    HEALTH, FAILURE, DOWNTIME, COST = 0, 1, 4, 5

    if action == 0:  # Do nothing
        next_state[HEALTH] = apply_probabilistic_change(next_state[HEALTH], 0.1, 0.6)
        next_state[FAILURE] = apply_probabilistic_change(next_state[FAILURE], 0.7, 0.1)
        next_state[DOWNTIME] = apply_probabilistic_change(next_state[DOWNTIME], 0.6, 0.1)
        next_state[COST] += 0.05

    elif action == 1:  # Preventive
        next_state[HEALTH] = apply_probabilistic_change(next_state[HEALTH], 0.7, 0.1)
        next_state[FAILURE] = apply_probabilistic_change(next_state[FAILURE], 0.1, 0.7)
        next_state[DOWNTIME] = apply_probabilistic_change(next_state[DOWNTIME], 0.1, 0.6)
        next_state[COST] -= 0.03

    elif action == 2:  # Corrective
        if np.random.rand() < 0.9:
            next_state[HEALTH] = 0.9
            next_state[FAILURE] = 0.1
            next_state[DOWNTIME] = 0.1
        else:
            next_state[HEALTH] = 0.6
            next_state[FAILURE] = 0.3
            next_state[DOWNTIME] = 0.3
        next_state[COST] += 0.08

    next_state += np.random.normal(0, 0.01, size=len(state))
    return np.clip(next_state, 0, 1)

In [14]:
def compute_reward(state, action, row):
    health = state[0]
    downtime = state[4]

    profit = row["Net_Profit"]
    downtime_cost = row["DownTime_Cost"]

    action_cost = 0
    if action == 1:
        action_cost = 500
    elif action == 2:
        action_cost = 2000

    reward = (
        0.5 * profit
        + 30 * health
        - 40 * downtime
        - 0.2 * downtime_cost
        - action_cost
    )

    return reward

In [15]:
episode_rewards = []

for ep in range(episodes):
    idx = np.random.randint(len(X_train))
    state = X_train[idx]
    row = df.iloc[idx]

    total_reward = 0

    for step in range(50):
        action = choose_action(state)

        next_state = environment_step(state, action)
        reward = compute_reward(state, action, row)

        q_current = np.dot(weights[action], state)
        q_next = np.max(get_q_values(next_state))

        td_target = reward + gamma * q_next
        td_error = td_target - q_current

        weights[action] += alpha * td_error * state

        state = next_state
        total_reward += reward

    episode_rewards.append(total_reward)

    if ep % 50 == 0:
        print(f"Episode {ep}, Reward: {total_reward:.2f}")

Episode 0, Reward: 55600.67
Episode 50, Reward: 4626.72
Episode 100, Reward: 41226.04
Episode 150, Reward: -147125.77
Episode 200, Reward: 363267.50
Episode 250, Reward: 105151.16
Episode 300, Reward: 27482.33
Episode 350, Reward: 270533.36
Episode 400, Reward: 221451.55
Episode 450, Reward: 308380.24


In [16]:
def evaluate_policy():
    state = X_test[0]
    total_reward = 0

    for _ in range(50):
        action = np.argmax(get_q_values(state))
        state = environment_step(state, action)
        total_reward += compute_reward(state, action, df.iloc[0])

    return total_reward

print("Policy Reward:", evaluate_policy())

Policy Reward: 197974.74681993492


In [17]:
sample = X_test[0]
q_vals = get_q_values(sample)

actions = ["Do Nothing", "Preventive", "Corrective"]

print("Q-values:", q_vals)
print("Best Action:", actions[np.argmax(q_vals)])

Q-values: [28546.30184592 25796.85319546 24004.84246678]
Best Action: Do Nothing
